In [ ]:
library(here)
library(maplet)
library(dplyr)
library(purrr)
library(glmnet)
library(ggpubr)

# set repo path
repo <- here()
renv::activate(project = repo)

In [ ]:
# load maplet object
D <- readRDS(here('data', 'preprocessed_venous_metabolon.RDS'))

# extract clinical data
clinical <- D %>% colData() %>% as.data.frame()

In [ ]:
# filter out healthy controls and comparators - select PH only 
D <- D %>% 
    mt_modify_filter_samples(filter = group_name == 'PH')

# Multivariable Modeling

## Model Inspection

### Metabolic GLS 6`

In [ ]:
# specify clinical parameter 
clin_param <- 'RVGLOB6n'
short_hand <- 'GLS 6`'

# subset maplet object by clinical parameter
D_model <- D %>% 
    mt_modify_filter_samples(filter = !is.na(!!sym(clin_param))) 

D_model

In [ ]:
# prepare data for model training
df_y <- colData(D_model) %>% as.data.frame() %>% select(all_of(clin_param)) %>% as.matrix()
df_x <- assay(D_model) %>% t() %>% as.matrix() %>% scale()

In [ ]:
set.seed(123) # for reproducibility
# train model using lasso regression
gls_model <- cv.glmnet(df_x, df_y, family = "gaussian", type.measure = "mse", alpha = 1) 

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 4) 

plot(gls_model, main = paste0("CV for ", short_hand, " Lasso Model\n\n"))

# print number of mets selected for this model
cat("Number of features selected for model:", sum(coef(gls_model, s = "lambda.min") != 0) - 1, "\n") # (-1 for intercept)

In [ ]:
# store final model and export
final_model <- coef(gls_model$glmnet.fit, s = gls_model$lambda.min) %>% as.matrix() %>% as.data.frame() %>% 
    # reformat column names
    dplyr::rename(coef = 1) %>% tibble::rownames_to_column('Feature') 

write.csv(final_model, here('metrv_models','gls_model_coef.csv'), row.names = FALSE)

### Metabolic FAC

In [ ]:
# specify clinical parameter 
clin_param <- 'RVFACn'
short_hand <- 'FAC'

# subset maplet object by clinical parameter
D_model <- D %>% 
    mt_modify_filter_samples(filter = !is.na(!!sym(clin_param))) 

D_model

In [ ]:
# prepare data for model training
df_y <- colData(D_model) %>% as.data.frame() %>% select(all_of(clin_param)) %>% as.matrix()
df_x <- assay(D_model) %>% t() %>% as.matrix() %>% scale()

In [ ]:
set.seed(123) # for reproducibility
# train model using lasso regression
fac_model <- cv.glmnet(df_x, df_y, family = "gaussian", type.measure = "mse", alpha = 1) 

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 4) 

plot(fac_model, main = paste0("CV for ", short_hand, " Lasso Model\n\n"))

# print number of mets selected for this model
cat("Number of features selected for model:", sum(coef(fac_model, s = "lambda.min") != 0) - 1, "\n") # (-1 for intercept)

In [ ]:
# store final model and export
final_model <- coef(fac_model$glmnet.fit, s = fac_model$lambda.min) %>% as.matrix() %>% as.data.frame() %>% 
    # reformat column names
    dplyr::rename(coef = 1) %>% tibble::rownames_to_column('Feature') 

write.csv(final_model, here('metrv_models','fac_model_coef.csv'), row.names = FALSE)

### Metabolic RVEF

In [ ]:
# specify clinical parameter 
clin_param <- 'mri_RVEF'
short_hand <- 'RVEF'

# subset maplet object by clinical parameter
D_model <- D %>% 
    mt_modify_filter_samples(filter = !is.na(!!sym(clin_param))) 

D_model

In [ ]:
# prepare data for model training
df_y <- colData(D_model) %>% as.data.frame() %>% select(all_of(clin_param)) %>% as.matrix()
df_x <- assay(D_model) %>% t() %>% as.matrix() %>% scale()

In [ ]:
set.seed(123) # for reproducibility
# train model using lasso regression
rvef_model <- cv.glmnet(df_x, df_y, family = "gaussian", type.measure = "mse", alpha = 1) 

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 4) 

plot(rvef_model, main = paste0("CV for ", short_hand, " Lasso Model\n\n"))

# print number of mets selected for this model
cat("Number of features selected for model:", sum(coef(rvef_model, s = "lambda.min") != 0) - 1, "\n") # (-1 for intercept)

In [ ]:
# store final model and export
final_model <- coef(rvef_model$glmnet.fit, s = rvef_model$lambda.min) %>% as.matrix() %>% as.data.frame() %>% 
    # reformat column names
    dplyr::rename(coef = 1) %>% tibble::rownames_to_column('Feature') 

write.csv(final_model, here('metrv_models','rvef_model_coef.csv'), row.names = FALSE)

## Out-of-Fold Model Evaluation
We'll now proceed to use lasso regualarized models to generate estimates for these parameters. This requires a more thorough 'train/test' setup. An alternative approach to doing this is to to create out-of-sample predictions using cross-validation: 

In short, we create $K$ folds (splits) of the data. For each fold, we train on the other 
$𝐾−1$ folds and then predict on the held-out fold. We store those predictions in the correct location in a vector (or data frame).

### Metabolic GLS 6`

In [ ]:
# specify clinical parameter 
clin_param <- 'RVGLOB6n'
short_hand <- 'GLS 6`'

# subset maplet object by clinical parameter
D_model <- D %>% 
    mt_modify_filter_samples(filter = !is.na(!!sym(clin_param))) 

In [ ]:
# prepare data for model training
df_y <- colData(D_model) %>% as.data.frame() %>% select(all_of(clin_param)) %>% as.matrix()
df_x <- assay(D_model) %>% t() %>% as.matrix() %>% scale()

In [ ]:
set.seed(123)  # For reproducibility
K <- 5 # number of folds for oof prediction 

# Make a vector that tells you which fold each participant belongs to
fold_assignments <- sample(rep_len(1:K, nrow(df_x)))

# define partitioning for each sample's out-of-sample prediction
cv_preds <- rep(NA, nrow(df_x))  

In [ ]:
# for loop to store predictions
for (fold in 1:K) {
  # Identify train vs test indices
  test_idx  <- which(fold_assignments == fold)
  train_idx <- setdiff(seq_len(nrow(df_x)), test_idx)
  
  # Subset x and y to training rows
  x_train <- df_x[train_idx, ]
  y_train <- df_y[train_idx]
  
  # Fit CV glmnet on training fold
  gls_model_fold <- cv.glmnet(
    x_train,
    y_train,
    family = "gaussian",
    type.measure = "mse",
    alpha = 1
  )
  
  # Predict on the held-out (test) fold
  x_test <- df_x[test_idx, ]
  
  fold_preds <- predict(
    gls_model_fold$glmnet.fit,
    newx = x_test,
    s    = gls_model_fold$lambda.min
  )
  
  # Store fold predictions in the overall vector
  cv_preds[test_idx] <- fold_preds
}


In [ ]:
# Turn cv_preds into a data.frame for merging
cv_pred_df <- data.frame(
  SAMP_ID   = rownames(df_x),     # The unique sample ID from df_x
  holdout_RVGLOB6n   = cv_preds            # The cross-validated prediction
)

# Then join to clin_temp:
clin_joined <- clinical %>%
  tibble::rownames_to_column('SAMP_ID') %>%
  dplyr::inner_join(cv_pred_df, by = 'SAMP_ID') %>%
  tibble::column_to_rownames('SAMP_ID')

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 4) 

# plot predicted vs actual values
lims <- range(c(clin_joined[[clin_param]], clin_joined[[paste0('holdout_', clin_param)]]), na.rm = TRUE) # get axis limits for same x and y range

ggplot(clin_joined, aes(y = !!sym(paste0('holdout_', clin_param)), x = !!sym(clin_param))) + 
    geom_point() + theme_bw() + 
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") + 
    # plot best fit line
    geom_smooth(method = "lm", se = TRUE, color = "blue", lwd = 0.5) +
    # add pearson correlation coefficient
    stat_cor(method = "pearson", label.x = lims[1], label.y = lims[2]) +
    labs(title = paste0("Predicted vs Actual Values for ", short_hand), y = paste0("Predicted ", short_hand), x = paste0("Actual ", short_hand)) +
    coord_cartesian(xlim = lims, ylim = lims)

### Metabolic FAC

In [ ]:
# specify clinical parameter 
clin_param <- 'RVFACn'
short_hand <- 'FAC'

# subset maplet object by clinical parameter
D_model <- D %>% 
    mt_modify_filter_samples(filter = !is.na(!!sym(clin_param))) 

In [ ]:
# prepare data for model training
df_y <- colData(D_model) %>% as.data.frame() %>% select(all_of(clin_param)) %>% as.matrix()
df_x <- assay(D_model) %>% t() %>% as.matrix() %>% scale()

In [ ]:
set.seed(123)  # For reproducibility
K <- 5 # number of folds for oof prediction 

# Make a vector that tells you which fold each participant belongs to
fold_assignments <- sample(rep_len(1:K, nrow(df_x)))

# define partitioning for each sample's out-of-sample prediction
cv_preds <- rep(NA, nrow(df_x))  

In [ ]:
# for loop to store predictions
for (fold in 1:K) {
  # Identify train vs test indices
  test_idx  <- which(fold_assignments == fold)
  train_idx <- setdiff(seq_len(nrow(df_x)), test_idx)
  
  # Subset x and y to training rows
  x_train <- df_x[train_idx, ]
  y_train <- df_y[train_idx]
  
  # Fit CV glmnet on training fold
  fac_model_fold <- cv.glmnet(
    x_train,
    y_train,
    family = "gaussian",
    type.measure = "mse",
    alpha = 1
  )
  
  # Predict on the held-out (test) fold
  x_test <- df_x[test_idx, ]
  
  fold_preds <- predict(
    fac_model_fold$glmnet.fit,
    newx = x_test,
    s    = fac_model_fold$lambda.min
  )
  
  # Store fold predictions in the overall vector
  cv_preds[test_idx] <- fold_preds
}


In [ ]:
# Turn cv_preds into a data.frame for merging
cv_pred_df <- data.frame(
  SAMP_ID   = rownames(df_x),     # The unique sample ID from df_x
  holdout_RVFACn   = cv_preds            # The cross-validated prediction
)

# Then join to clin_temp:
clin_joined <- clinical %>%
  tibble::rownames_to_column('SAMP_ID') %>%
  dplyr::inner_join(cv_pred_df, by = 'SAMP_ID') %>%
  tibble::column_to_rownames('SAMP_ID')

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 4) 

# plot predicted vs actual values
lims <- range(c(clin_joined[[clin_param]], clin_joined[[paste0('holdout_', clin_param)]]), na.rm = TRUE) # get axis limits for same x and y range

ggplot(clin_joined, aes(y = !!sym(paste0('holdout_', clin_param)), x = !!sym(clin_param))) + 
    geom_point() + theme_bw() + 
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") + 
    # plot best fit line
    geom_smooth(method = "lm", se = TRUE, color = "blue", lwd = 0.5) +
    # add pearson correlation coefficient
    stat_cor(method = "pearson", label.x = lims[1], label.y = lims[2]) +
    labs(title = paste0("Predicted vs Actual Values for ", short_hand), y = paste0("Predicted ", short_hand), x = paste0("Actual ", short_hand)) +
    coord_cartesian(xlim = lims, ylim = lims)

### Metabolic RVEF

In [ ]:
# specify clinical parameter 
clin_param <- 'mri_RVEF'
short_hand <- 'RVEF'

# subset maplet object by clinical parameter
D_model <- D %>% 
    mt_modify_filter_samples(filter = !is.na(!!sym(clin_param))) 

In [ ]:
# prepare data for model training
df_y <- colData(D_model) %>% as.data.frame() %>% select(all_of(clin_param)) %>% as.matrix()
df_x <- assay(D_model) %>% t() %>% as.matrix() %>% scale()

In [ ]:
set.seed(123)  # For reproducibility
K <- 5 # number of folds for oof prediction 

# Make a vector that tells you which fold each participant belongs to
fold_assignments <- sample(rep_len(1:K, nrow(df_x)))

# define partitioning for each sample's out-of-sample prediction
cv_preds <- rep(NA, nrow(df_x))  

In [ ]:
# for loop to store predictions
for (fold in 1:K) {
  # Identify train vs test indices
  test_idx  <- which(fold_assignments == fold)
  train_idx <- setdiff(seq_len(nrow(df_x)), test_idx)
  
  # Subset x and y to training rows
  x_train <- df_x[train_idx, ]
  y_train <- df_y[train_idx]
  
  # Fit CV glmnet on training fold
  rvef_model_fold <- cv.glmnet(
    x_train,
    y_train,
    family = "gaussian",
    type.measure = "mse",
    alpha = 1
  )
  
  # Predict on the held-out (test) fold
  x_test <- df_x[test_idx, ]
  
  fold_preds <- predict(
    rvef_model_fold$glmnet.fit,
    newx = x_test,
    s    = rvef_model_fold$lambda.min
  )
  
  # Store fold predictions in the overall vector
  cv_preds[test_idx] <- fold_preds
}


In [ ]:
# Turn cv_preds into a data.frame for merging
cv_pred_df <- data.frame(
  SAMP_ID   = rownames(df_x),     # The unique sample ID from df_x
  holdout_mri_RVEF   = cv_preds            # The cross-validated prediction
)

# Then join to clin_temp:
clin_joined <- clinical %>%
  tibble::rownames_to_column('SAMP_ID') %>%
  dplyr::inner_join(cv_pred_df, by = 'SAMP_ID') %>%
  tibble::column_to_rownames('SAMP_ID')

In [ ]:
clin_joined %>% select(mri_RVEF, holdout_mri_RVEF) %>% summary()

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 4) 

# plot predicted vs actual values
lims <- range(c(clin_joined[[clin_param]], clin_joined[[paste0('holdout_', clin_param)]]), na.rm = TRUE) # get axis limits for same x and y range

ggplot(clin_joined, aes(y = !!sym(paste0('holdout_', clin_param)), x = !!sym(clin_param))) + 
    geom_point() + theme_bw() + 
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") + 
    # plot best fit line
    geom_smooth(method = "lm", se = TRUE, color = "blue", lwd = 0.5) +
    # add pearson correlation coefficient
    stat_cor(method = "pearson", label.x = lims[1], label.y = lims[2]) +
    labs(title = paste0("Predicted vs Actual Values for ", short_hand), y = paste0("Predicted ", short_hand), x = paste0("Actual ", short_hand)) +
    coord_cartesian(xlim = lims, ylim = lims)